In [3]:
using StochasticRounding, LinearAlgebra

A = randn(Float32sr, 50, 50)
b = randn(Float32sr, 50)

x1 = A \ b
x2 = A \ b

println(norm(Float32.(x1) - Float32.(x2)))


0.0018383467


In [16]:
using StochasticRounding, Statistics

Tsr = Float16sr
N = 20000
ε = Tsr(1e-5)

function accumulate_sr(N, ε)
    s = Tsr(0)
    for _ in 1:N
        s += ε
    end
    return Float32(s)
end

vals = [accumulate_sr(N, ε) for _ in 1:20]
println("mean = ", mean(vals))
println("std  = ", std(vals))


mean = 0.1987793
std  = 0.0031438859


In [27]:
using StochasticRounding, LinearAlgebra

A = randn(Float32sr, 80, 80)
b = randn(Float32sr, 80)

x1 = A \ b
x2 = A \ b
println("‖x1-x2‖ = ", norm(Float32.(x1) - Float32.(x2)))

# Compare to "final only" simulation: do deterministic solve, then one SR rounding
A_d = Float32.(A); b_d = Float32.(b)
x_det = A_d \ b_d
r1 = stochastic_round.(Float32sr, x_det)
r2 = stochastic_round.(Float32sr, x_det)
println("‖r1-r2‖ (final-only) = ", norm(Float32.(r1) - Float32.(r2)))


‖x1-x2‖ = 0.001002475
‖r1-r2‖ (final-only) = 0.0


In [29]:
using LinearAlgebra

println(typeof(A))
F = factorize(A)  # looks at type/properties of A
println(typeof(F))  # e.g., LinearAlgebra.LU{Float32sr,...}


Matrix{Float32sr}
LU{Float32sr, Matrix{Float32sr}, Vector{Int64}}


In [40]:
using StochasticRounding
using BenchmarkTools
using LinearAlgebra

A = rand(Float16sr, 512, 512)
B = rand(Float16sr, 512, 512)

@time for i in 1:10
    C = A * B        # every multiply triggers SR
end

A = Float16.(A)
B = Float16.(B)

@time for i in 1:10
    C = A * B
end


  8.034111 seconds (30 allocations: 5.001 MiB)
  0.172353 seconds (1.73 M allocations: 94.499 MiB, 9.26% gc time, 71.65% compilation time)


In [41]:
using StochasticRounding
using LinearAlgebra

A = rand(Float16sr, 5, 5)
b = rand(Float16sr, 5)

A_low = Float16.(A)
b_low = Float16.(b)

x = A \ b 

x_low = A_low \ b_low

println(typeof(x))
println(x)
println()
println(typeof(x_low))
println(x_low)

Vector{Float16sr}
Float16sr[Float16(0.001314), Float16(0.345), Float16(0.1282), Float16(0.6064), Float16(0.3674)]

Vector{Float16}
Float16[0.002104, 0.3452, 0.1278, 0.6064, 0.3662]
